# Climate change on TikTok: released-data analysis

[Paper](https://arxiv.org/html/2505.03813) · [Zenodo release](https://zenodo.org/records/15131067). The release contains video topic/hashtag records, creator topic vectors, and a follower network, but no raw video, comment text, or engagement counts. This is a keyword- and language-filtered collection, not a random TikTok sample.

## Local files and setup

**Purpose:** Confirm which public files are actually present on this computer. File presence does not imply that raw media or comment rows are included.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'downloaded_datasets').exists() and (ROOT.parent / 'downloaded_datasets').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'downloaded_datasets' / 'climate_change'
print('Local data folder:', DATA.resolve())
if DATA.exists():
    display(pd.DataFrame([{'file': p.name, 'size_MB': round(p.stat().st_size / 1e6, 2)}
                          for p in sorted(DATA.iterdir()) if p.is_file()]))
else:
    print('Local data folder is missing. Download the files first.')

Local data folder: /Users/wang/Documents/tiktok/downloaded_datasets/climate_change


,file,size_MB
0,READ-ME.md,0.00
1,network.csv,5.69
2,profiles.csv,2.75
3,topics.json,0.00
4,videos.json,180.38


## 1. File units and paper counts

**Purpose:** Identify the unit of each file, check duplicates, and inspect the actual format of `topics.json`.

In [2]:
import json
import numpy as np
from collections import Counter
with (DATA / 'videos.json').open(encoding='utf-8') as handle:
    videos = pd.DataFrame(json.load(handle))
profiles = pd.read_csv(DATA / 'profiles.csv')
network = pd.read_csv(DATA / 'network.csv')
topics = pd.read_csv(DATA / 'topics.json')  # Despite the extension, the released file is CSV text.
display(pd.DataFrame([
    {'file': 'videos.json', 'rows': len(videos), 'unit': 'video record'},
    {'file': 'profiles.csv', 'rows': len(profiles), 'unit': 'creator topic vector'},
    {'file': 'network.csv', 'rows': len(network), 'unit': 'directed following edge'},
    {'file': 'topics.json', 'rows': len(topics), 'unit': 'topic index (CSV text)'},
]))
print('Video columns:', videos.columns.tolist())
print('Unique video IDs:', videos.video_id.nunique(), 'unique author IDs:', videos.author_id.nunique())
print('Unique profile users:', profiles.user_ID.nunique())
print('Topic labels:', len(topics), 'includes Outliers:', topics.Label.eq('Outliers').any())

,file,rows,unit
0,videos.json,590361,video record
1,profiles.csv,14047,creator topic vector
2,network.csv,161891,directed following edge
3,topics.json,27,topic index (CSV text)


Video columns: ['author_id', 'video_id', 'hashtags', 'topic']
Unique video IDs: 590360 unique author IDs: 18041
Unique profile users: 14047
Topic labels: 27 includes Outliers: True


**What this shows:** The 590,361 video rows match the paper's filtered set, but only 590,360 video IDs are unique. There are 18,041 video author IDs and 14,047 profile users; these are not identical populations.

## 2. Topic distribution

**Purpose:** Identify the largest published topic groups and the unassigned share.

In [ ]:
display(videos.topic.value_counts(dropna=False).head(15).rename_axis('topic').to_frame('video_rows'))
print('Outliers share:', f'{100 * videos.topic.eq("Outliers").mean():.2f}%')
print('Distinct observed labels:', videos.topic.nunique(), 'topic-index labels:', len(topics))

,video_rows
topic,
Outliers,267718
Lifestyle,80861
Fashion,38869
Healthy Cooking,36962
Travel,27752
Hobby and Living,19340
Oceans,16726
Trump,11698
Sustainability,10680


Outliers share: 45.35%
Distinct observed labels: 27 topic-index labels: 27


**What this shows:** 267,718 rows (45.35%) are `Outliers`, meaning no specific topic was assigned by the model. This is not a climate opinion. Lifestyle is the largest named topic (80,861 rows).

## 3. Videos per creator

**Purpose:** Check whether a few active creators dominate the video-level sample.

In [4]:
per_creator = videos.groupby('author_id').size()
display(per_creator.describe(percentiles=[.25, .5, .75, .9, .99]).to_frame('video_rows_per_author'))
top_n = max(1, int(np.ceil(len(per_creator) * .01)))
print('Share of videos from most prolific 1% of authors (%):',
      round(100 * per_creator.nlargest(top_n).sum() / len(videos), 2))
print('Video author IDs also in profiles:', len(set(videos.author_id) & set(profiles.user_ID)))

,video_rows_per_author
count,18041.000000
mean,32.723297
std,50.823777
min,1.000000
25%,3.000000
50%,14.000000
75%,41.000000
90%,86.000000
99%,247.000000
max,717.000000


Share of videos from most prolific 1% of authors (%): 10.32
Video author IDs also in profiles: 14047


**What this shows:** The median is 14 videos per author, the mean is 32.72, and the top 1% of authors supply 10.32% of rows. These figures describe the selected collection, not all TikTok creators.

## 4. Hashtags

**Purpose:** Count author-provided hashtags per video and list common tags.

In [5]:
tag_lists = videos.hashtags.apply(lambda value: value if isinstance(value, list) else [])
tag_counts = tag_lists.map(len)
display(tag_counts.describe(percentiles=[.25, .5, .75, .9, .99]).to_frame('hashtags_per_video'))
tags = Counter(str(tag).lower() for row in tag_lists for tag in row)
display(pd.Series(dict(tags.most_common(15)), name='video_occurrences').to_frame())
print('Videos with no hashtags:', int(tag_counts.eq(0).sum()))

,hashtags_per_video
count,590361.000000
mean,15.497026
std,20.953116
min,0.000000
25%,6.000000
50%,10.000000
75%,18.000000
90%,30.000000
99%,125.000000
max,673.000000


,video_occurrences
fyp,153012
foryou,61175
foryoupage,48629
fypシ,41724
viral,39563
love,32011
trending,26563
tiktok,24291
nature,22085
funny,18882


Videos with no hashtags: 20008


**What this shows:** The median is 10 hashtags; 20,008 rows have none. `fyp` occurs 153,012 times. A broad reach tag does not establish a video's position on climate change.

## 5. Follower-network structure

**Purpose:** Calculate nodes and directed edges, and test whether network names can be joined to video author IDs.

In [6]:
nodes = set(network.source.astype(str)) | set(network.target.astype(str))
out_degree = network.groupby('source').size()
in_degree = network.groupby('target').size()
display(pd.DataFrame([{'edge_rows': len(network),
                       'unique_directed_edges': len(network.drop_duplicates(['source', 'target'])),
                       'unique_nodes': len(nodes), 'mean_outgoing_edges_per_node': len(network) / len(nodes)}]))
display(pd.DataFrame({'out_degree': out_degree.describe(percentiles=[.5, .9, .99]),
                      'in_degree': in_degree.describe(percentiles=[.5, .9, .99])}))
print('Direct overlap: numeric video author IDs vs network source names:',
      len(set(videos.author_id.astype(str)) & set(network.source.astype(str))))

,edge_rows,unique_directed_edges,unique_nodes,mean_outgoing_edges_per_node
0,161891,161891,13867,11.674551


,out_degree,in_degree
count,12448.000000,10173.000000
mean,13.005382,15.913791
std,23.094697,43.701726
min,1.000000,1.000000
50%,5.000000,4.000000
90%,32.000000,35.000000
99%,112.000000,177.120000
max,424.000000,974.000000


Direct overlap: numeric video author IDs vs network source names: 0


**What this shows:** There are 161,891 unique directed edges and 13,867 nodes. The network uses username-like strings, while videos use numeric IDs; direct overlap is zero, so these files cannot safely be joined as-is.

## 6. Creator topic vectors

**Purpose:** Check vector dimension and missing-information vectors before finding the dominant topic.

In [7]:
import ast
vectors = profiles.profile.map(ast.literal_eval)
lengths = vectors.map(len)
print('Vector lengths:', lengths.value_counts().to_dict())
matrix = np.array(vectors[lengths.eq(26)].tolist(), dtype=float)
print('Quantiles of vector sums:', np.round(np.quantile(matrix.sum(axis=1), [0, .25, .5, .75, 1]), 3))
nonzero = matrix.sum(axis=1) > 0
print('All-zero vectors:', int((~nonzero).sum()))
labels_by_id = dict(zip(topics.Topic_id, topics.Label))
dominant = pd.Series([labels_by_id.get(int(index), 'unknown') for index in matrix[nonzero].argmax(axis=1)])
display(dominant.value_counts().head(12).rename_axis('largest_vector_topic').to_frame('creators'))

Vector lengths: {26: 14047}
Quantiles of vector sums: [0. 1. 1. 1. 1.]
All-zero vectors: 650


,creators
largest_vector_topic,
Lifestyle,3778
Fashion,2032
Healthy Cooking,1207
Travel,790
Oceans,712
Hobby and Living,675
Trump,562
Animals,512
Climate Activism,495


**What this shows:** All published vectors have 26 elements; 650 are all zero and are excluded from dominant-topic analysis. Lifestyle leads among 3,778 nonzero profiles. A topic vector is not an attitude measure.

## Findings and limits

- `videos.json` contains **590,361** rows, **590,360** unique IDs, and **18,041** distinct author IDs; it lacks comment counts and raw media.
- **267,718** rows (45.35%) are topic-model `Outliers`; these are not a climate stance.
- The author-level video median is **14**, mean **32.72**; the most prolific 1% supply **10.32%** of video rows.
- `fyp` occurs **153,012** times as a hashtag; generic reach tags remain in the release.
- The follower network has **161,891** distinct directed edges among **13,867** nodes, but its names do not directly match numeric video author IDs.
- **650** creator topic vectors are all-zero. The file named `topics.json` is actually CSV text with 26 named topics plus `Outliers`.